In [1]:
from astrovision.plot.plot_utils import make_mosaic
from astrovision.data import SatelliteImage, SegmentationLabeledSatelliteImage
import s3fs
import os
import geopandas as gpd
from pyproj import CRS
import matplotlib.pyplot as plt
import pandas as pd
import folium
import numpy as np
import math
import ipywidgets as widgets
from ipywidgets import interact

from IPython.display import clear_output

In [2]:
fs = s3fs.S3FileSystem(client_kwargs={"endpoint_url": "https://" + "minio.lab.sspcloud.fr"})
out  = fs.download(rpath="projet-slums-detection/ilots", lpath="ilots", recursive=True)
out = fs.download(rpath="projet-slums-detection/data-prediction/PLEIADES/MAYOTTE", lpath="pred_mayotte", recursive=True)

pred_2020 = gpd.read_file('pred_mayotte/2020/predictions.gpkg')
pred_2023 = gpd.read_file('pred_mayotte/2023/predictions.gpkg')
ilots = gpd.read_file("ilots/ilots.gpkg")
tgt_crs = CRS.from_epsg(4471) # mayotte 

pred_2020 = pred_2020.to_crs(tgt_crs)
pred_2023 = pred_2023.to_crs(tgt_crs)

pred_2020["area"] = pred_2020.area
pred_2023["area"] = pred_2023.area

ilots = ilots.to_crs(tgt_crs)
ilotsMayotte = ilots[ilots["ident_ilot"].str.startswith('976')]
ilotsMayotte["area"] = ilotsMayotte.geometry.area

/opt/mamba/lib/python3.11/site-packages/geopandas/geodataframe.py:1525: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


In [3]:
intersection_2020 = gpd.overlay(pred_2020, ilotsMayotte, how='intersection')
intersection_2023 = gpd.overlay(pred_2023, ilotsMayotte, how='intersection')

In [7]:
intersection_2020["area_pred_inter_ilot"] = intersection_2020['geometry'].area
intersection_2023["area_pred_inter_ilot"] = intersection_2020['geometry'].area

aire_inter_2020 = intersection_2020.groupby('ident_ilot')['area_pred_inter_ilot'].sum()
aire_inter_2023 = intersection_2023.groupby('ident_ilot')['area_pred_inter_ilot'].sum()

comp_area = pd.merge(aire_inter_2020,aire_inter_2023,how = 'outer', on = 'ident_ilot', suffixes=('_2020', '_2023'))

comp_area["diff_area"] = comp_area["area_pred_inter_ilot_2023"] - comp_area["area_pred_inter_ilot_2020"]
comp_area["evol_pct_area"] = round((comp_area["diff_area"]/comp_area["area_pred_inter_ilot_2020"])*100,2)

In [21]:
def filtre_compacite (table, seuil_compacite):
    table['compacite'] = (4 * math.pi * table.area) / (table.length**2)
    table_filtree = table[table['compacite'] > seuil_compacite]
    return table_filtree

In [48]:
def filtre_taille (table, seuil_taille):
    if seuil_taille != 0:
        table_triee = table.sort_values(by='area_1')
        decile_seuil = np.percentile(table_triee['area_1'], seuil_taille)
        polygone_decile = table_triee[table_triee['area_1'] <= decile_seuil].iloc[-1]
        table_filtree = table[table['area_1'] > polygone_decile['area_1']]
    else : 
        table_filtree = table
    return table_filtree